In [27]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 4. Selección de características y normalización (MEJORADO)
# ===================================================================

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
import pickle
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR  = 'ml_features_grouped'
OUTPUT_DIR = 'ml_normalized_grouped'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================================================================
# 0. VERIFICAR ARCHIVOS
# ===================================================================
print("\n" + "="*60)
print("PASO 4: SELECCIÓN Y NORMALIZACIÓN")
print("="*60)

print("\n📁 VERIFICANDO ARCHIVOS DE ENTRADA...")
files_needed = ['X_train.xlsx', 'X_test.xlsx', 'y_train_regression.xlsx', 'y_test_regression.xlsx', 'feature_names.pkl']
missing = []
for f in files_needed:
    path = os.path.join(INPUT_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"   ✅ {f} ({size} bytes)")
    else:
        print(f"   ❌ {f} (NO EXISTE)")
        missing.append(f)

if missing:
    print(f"\n❌ ERROR: Faltan archivos en '{INPUT_DIR}/'")
    print(f"   Ejecuta primero el PASO 3 completo")
    raise SystemExit("Faltan archivos de entrada")

# ===================================================================
# 1. CARGAR DATOS
# ===================================================================
print("\n" + "="*60)
print("1. LOADING DATA")
print("="*60)

# Cargar datos con pandas para identificar tipos
X_train_df = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx'))
X_test_df = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx'))
y_train = pd.read_excel(os.path.join(INPUT_DIR, 'y_train_regression.xlsx')).values.ravel()
y_test = pd.read_excel(os.path.join(INPUT_DIR, 'y_test_regression.xlsx')).values.ravel()

with open(os.path.join(INPUT_DIR, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)

print(f"\n   X_train original: {X_train_df.shape}")
print(f"   X_test original:  {X_test_df.shape}")
print(f"   y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"   Features totales: {len(feature_names)}")

# ===================================================================
# 1.5 LIMPIAR COLUMNAS NO NUMÉRICAS
# ===================================================================
print("\n" + "="*60)
print("1.5 CLEANING NON-NUMERIC COLUMNS")
print("="*60)

# Identificar columnas numéricas
numeric_cols = []
non_numeric_cols = []

for col in X_train_df.columns:
    if pd.api.types.is_numeric_dtype(X_train_df[col]):
        numeric_cols.append(col)
    else:
        non_numeric_cols.append(col)

print(f"\n   Columnas numéricas: {len(numeric_cols)}")
print(f"   Columnas no numéricas: {len(non_numeric_cols)}")

if non_numeric_cols:
    print(f"\n   ⚠️  Eliminando columnas no numéricas:")
    for col in non_numeric_cols:
        print(f"      - {col}")
    
    # Filtrar solo columnas numéricas
    X_train_df = X_train_df[numeric_cols]
    X_test_df = X_test_df[numeric_cols]
    
    # Actualizar feature_names
    feature_names = numeric_cols
    print(f"\n   ✅ Features numéricos: {len(feature_names)}")

# Convertir a numpy
X_train = X_train_df.values
X_test = X_test_df.values

print(f"\n   X_train final (numérico): {X_train.shape}")
print(f"   X_test final (numérico):  {X_test.shape}")

# ===================================================================
# 2. SELECCIÓN DE CARACTERÍSTICAS (MUTUAL INFORMATION)
# ===================================================================
print("\n" + "="*60)
print("2. FEATURE SELECTION (Mutual Information)")
print("="*60)

# Verificar que hay datos suficientes
if X_train.shape[1] == 0:
    raise ValueError("No hay columnas numéricas para seleccionar features")

# Calcular scores
try:
    scores = mutual_info_regression(X_train, y_train, random_state=42)
except Exception as e:
    print(f"\n   ❌ Error en mutual_info_regression: {e}")
    print("   Usando f_regression como alternativa...")
    from sklearn.feature_selection import f_regression
    scores, _ = f_regression(X_train, y_train)

df_scores = pd.DataFrame({
    'Feature': feature_names,
    'Score': scores
}).sort_values('Score', ascending=False)

# Determinar k - máximo 20 o número de features con score > 0.01
k = sum(s > 0.01 for s in scores)
k = min(k, 20)  # Máximo 20 features
k = min(k, X_train.shape[0] - 1)  # No más que muestras-1
k = min(k, X_train.shape[1])  # No más que features disponibles
if k < 1:
    k = min(5, X_train.shape[0] - 1, X_train.shape[1])

print(f"\n   k seleccionado: {k}")
print(f"   (features con score > 0.01, máximo 20)")

# Seleccionar
selector = SelectKBest(mutual_info_regression, k=k)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

selected_mask = selector.get_support()
selected_features = np.array(feature_names)[selected_mask].tolist()
discarded_features = np.array(feature_names)[~selected_mask].tolist()

# ============================================================
# MOSTRAR SELECCIONADAS
# ============================================================
print(f"\n   🟢 SELECTED ({len(selected_features)} features):")
print("   " + "-"*60)
for i, feat in enumerate(selected_features, 1):
    score = df_scores[df_scores['Feature'] == feat]['Score'].values[0]
    
    # Emojis según tipo
    marca = ""
    if 'Minutes_Since' in feat:
        marca = " 🕐 TIEMPO"
    elif 'CO2_Peak' in feat:
        marca = " 🟢 PICO CO₂"
    elif '_ratio_to_peak' in feat:
        marca = " 📊 RATIO"
    elif '_trend' in feat:
        marca = " 📈 TENDENCIA"
    elif 'CO2' in feat:
        marca = " 🟢 CO₂"
    elif 'Temperature' in feat:
        marca = " 🌡️ TEMP"
    elif 'Humidity' in feat:
        marca = " 💧 HUM"
    
    print(f"   {i:2d}. {feat:<50s} {score:.4f}{marca}")

# ============================================================
# MOSTRAR DESCARTADAS (primeras 10)
# ============================================================
if discarded_features:
    print(f"\n   🔴 DISCARDED ({len(discarded_features)} features) — mostrando primeras 10:")
    print("   " + "-"*60)
    for i, feat in enumerate(discarded_features[:10], 1):
        score = df_scores[df_scores['Feature'] == feat]['Score'].values[0]
        print(f"   {i:2d}. {feat:<50s} {score:.4f}")

# ============================================================
# RESUMEN
# ============================================================
print(f"\n   📊 RESUMEN:")
print(f"   Total features:    {len(feature_names)}")
print(f"   Seleccionadas:     {len(selected_features)}")
print(f"   Descartadas:       {len(discarded_features)}")

# ===================================================================
# 3. GRÁFICO TOP 20 FEATURES
# ===================================================================
print("\n   Generando gráfico TOP 20...")

top_n_plot = min(20, len(feature_names))
df_plot = df_scores.head(top_n_plot).copy()
df_plot['Status'] = np.where(df_plot['Feature'].isin(selected_features), 'Selected', 'Discarded')
df_plot = df_plot.sort_values('Score', ascending=True)

n_sel_plot = sum(df_plot['Status'] == 'Selected')
n_disc_plot = sum(df_plot['Status'] == 'Discarded')

fig, ax = plt.subplots(figsize=(14, 8))

def get_color(feat, status):
    if status == 'Discarded':
        return '#e74c3c'
    if 'Minutes_Since' in feat:
        return '#e74c3c'
    elif '_trend' in feat:
        return '#9b59b6'
    elif 'CO2' in feat:
        return '#2ecc71'
    elif 'Temperature' in feat or 'Temp_' in feat:
        return '#e67e22'
    elif 'Humidity' in feat:
        return '#3498db'
    elif 'Solar' in feat:
        return '#f39c12'
    else:
        return '#95a5a6'

bar_colors = [get_color(row['Feature'], row['Status']) for _, row in df_plot.iterrows()]

ax.barh(range(len(df_plot)), df_plot['Score'], color=bar_colors, edgecolor='white', linewidth=0.5)

if k <= top_n_plot:
    threshold_score = df_scores.iloc[k-1]['Score'] if k > 0 else 0
    ax.axvline(x=threshold_score, color='black', linestyle='--', linewidth=2, alpha=0.8,
               label=f'Threshold (top {k}) = {threshold_score:.4f}')

ax.set_yticks(range(len(df_plot)))
labels = [f[:52] + '...' if len(f) > 55 else f for f in df_plot['Feature']]
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Mutual Information Score', fontsize=13, fontweight='bold')
ax.set_title(f'Feature Selection — Aula 1.5 (TOP {top_n_plot})\n🟢 {n_sel_plot} Selected | 🔴 {n_disc_plot} Discarded (de {len(feature_names)} totales)',
             fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='🕐 Tiempo (Minutes_Since)'),
    Patch(facecolor='#9b59b6', label='📈 Tendencia (*_trend)'),
    Patch(facecolor='#2ecc71', label='🟢 CO₂'),
    Patch(facecolor='#e67e22', label='🌡️ Temperatura'),
    Patch(facecolor='#3498db', label='💧 Humedad'),
    Patch(facecolor='#f39c12', label='☀️ Solar'),
    Patch(facecolor='#e74c3c', label='🔴 Discarded', alpha=0.3),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right', ncol=2)
ax.grid(True, alpha=0.2, axis='x')

for i, (_, row) in enumerate(df_plot.iterrows()):
    ax.text(row['Score'] + 0.005, i, f"{row['Score']:.3f}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_selection_top20.png'), dpi=200, bbox_inches='tight')
plt.close()
print(f"   ✓ feature_selection_top20.png")

# ===================================================================
# 4. NORMALIZACIÓN
# ===================================================================
print("\n" + "="*60)
print("4. NORMALIZATION (RobustScaler)")
print("="*60)

# Usar RobustScaler (menos sensible a outliers)
scaler = RobustScaler()
X_train_norm = scaler.fit_transform(X_train_sel)
X_test_norm = scaler.transform(X_test_sel)

print(f"   Train → mean: {X_train_norm.mean():.6f}, std: {X_train_norm.std():.4f}")
print(f"   Test  → mean: {X_test_norm.mean():.6f},  std: {X_test_norm.std():.4f}")

# ===================================================================
# 5. GUARDAR
# ===================================================================
print("\n" + "="*60)
print("5. SAVING")
print("="*60)

# Guardar X (raw y normalizadas)
pd.DataFrame(X_train_sel, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_train.xlsx'), index=False)
pd.DataFrame(X_test_sel, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_test.xlsx'), index=False)
pd.DataFrame(X_train_norm, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_train_normalised.xlsx'), index=False)
pd.DataFrame(X_test_norm, columns=selected_features).to_excel(
    os.path.join(OUTPUT_DIR, 'X_test_normalised.xlsx'), index=False)

# Guardar y
pd.DataFrame({'Occupancy': y_train}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_train.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_test}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_test.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_train}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_train_regression.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_test}).to_excel(
    os.path.join(OUTPUT_DIR, 'y_test_regression.xlsx'), index=False)

# Guardar scores y metadata
df_scores.to_excel(os.path.join(OUTPUT_DIR, 'feature_scores_all.xlsx'), index=False)

with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(OUTPUT_DIR, 'selector.pkl'), 'wb') as f:
    pickle.dump(selector, f)
with open(os.path.join(OUTPUT_DIR, 'selected_features.pkl'), 'wb') as f:
    pickle.dump(selected_features, f)
with open(os.path.join(OUTPUT_DIR, 'feature_names.pkl'), 'wb') as f:
    pickle.dump(selected_features, f)

# ===================================================================
# 6. LISTAR ARCHIVOS GUARDADOS
# ===================================================================
print(f"\n   📁 Archivos guardados en '{OUTPUT_DIR}/':")
print("   " + "-"*40)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    icon = "📊" if fname.endswith('.png') else "📁"
    print(f"      {icon} {fname} ({size:.0f} KB)")

# ===================================================================
# 7. RESUMEN FINAL
# ===================================================================
print("\n" + "="*60)
print("6. SUMMARY")
print("="*60)
print(f"   Features originales:    {len(feature_names)}")
print(f"   Features seleccionadas: {len(selected_features)}")
print(f"   X_train final:          {X_train_norm.shape}")
print(f"   X_test final:           {X_test_norm.shape}")

# Variables de tiempo seleccionadas
time_features = [f for f in selected_features if 'Minutes_Since' in f]
if time_features:
    print(f"\n   🕐 Variables de TIEMPO seleccionadas ({len(time_features)}):")
    for tf in time_features:
        score = df_scores[df_scores['Feature'] == tf]['Score'].values[0]
        print(f"      {tf}: {score:.4f}")

print("\n" + "="*60)
print("✅ PASO 4 COMPLETADO!")
print("="*60)
print(f"\n📌 Ahora ejecuta el PASO 5 con los datos en '{OUTPUT_DIR}/'")


PASO 4: SELECCIÓN Y NORMALIZACIÓN

📁 VERIFICANDO ARCHIVOS DE ENTRADA...
   ✅ X_train.xlsx (64882 bytes)
   ✅ X_test.xlsx (33081 bytes)
   ✅ y_train_regression.xlsx (5706 bytes)
   ✅ y_test_regression.xlsx (5237 bytes)
   ✅ feature_names.pkl (2047 bytes)

1. LOADING DATA

   X_train original: (125, 66)
   X_test original:  (55, 66)
   y_train: (125,) | y_test: (55,)
   Features totales: 66

1.5 CLEANING NON-NUMERIC COLUMNS

   Columnas numéricas: 64
   Columnas no numéricas: 2

   ⚠️  Eliminando columnas no numéricas:
      - Start_Time
      - End_Time

   ✅ Features numéricos: 64

   X_train final (numérico): (125, 64)
   X_test final (numérico):  (55, 64)

2. FEATURE SELECTION (Mutual Information)

   k seleccionado: 20
   (features con score > 0.01, máximo 20)

   🟢 SELECTED (20 features):
   ------------------------------------------------------------
    1. Duration_min                                       0.4692
    2. Minutes_Since_DayStart                             0.9057 🕐